# Visit with Us — Wellness Tourism Package Purchase Prediction
### PG in AI/ML | End-to-End MLOps Project

**Business objective:** Predict whether a customer is likely to purchase the newly introduced Wellness Tourism Package before the marketing team contacts them.

This notebook demonstrates the complete ML lifecycle required for the project:
1. Data registration and validation
2. Data cleaning and preparation
3. Train/test workflow artifacts
4. Model experimentation and hyperparameter tuning
5. Evaluation and best-model persistence
6. MLOps/CI-CD repository structure
7. Streamlit deployment preparation

> **Target:** `ProdTaken` — 0 = No purchase, 1 = Purchase.


## 1. Data Registration
The master repository folder contains a `data/` subfolder. The dataset is loaded directly from `data/tourism.csv`. A validation step checks the expected schema and prints a data summary.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/tourism.csv")
EXPECTED_COLUMNS = [
    "CustomerID","ProdTaken","Age","TypeofContact","CityTier","DurationOfPitch",
    "Occupation","Gender","NumberOfPersonVisiting","NumberOfFollowups",
    "ProductPitched","PreferredPropertyStar","MaritalStatus","NumberOfTrips",
    "Passport","PitchSatisfactionScore","OwnCar","NumberOfChildrenVisiting",
    "Designation","MonthlyIncome"
]

assert DATA_PATH.exists(), f"Dataset not found at {DATA_PATH}"
df = pd.read_csv(DATA_PATH)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))

print("DATA REGISTRATION / VALIDATION")
print("=" * 55)
print(f"Dataset path: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Missing expected columns: {missing_columns}")
print(f"Extra columns: {extra_columns}")
assert not missing_columns, "Dataset schema validation failed."
print("\nSchema validation: PASSED")


DATA REGISTRATION / VALIDATION
Dataset path: data/tourism.csv
Rows: 4,128
Columns: 21
Missing expected columns: []
Extra columns: ['Unnamed: 0']

Schema validation: PASSED


In [2]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("Missing-value summary:")
display(df.isna().sum().to_frame("missing_count"))

print("Target distribution:")
display(df["ProdTaken"].value_counts().rename_axis("ProdTaken").to_frame("count"))

print("Descriptive summary:")
display(df.describe(include="all").T)


Data types:


,dtype
Unnamed: 0,int64
CustomerID,int64
ProdTaken,int64
Age,float64
TypeofContact,object
CityTier,int64
DurationOfPitch,float64
Occupation,object
Gender,object
NumberOfPersonVisiting,int64


Missing-value summary:


,missing_count
Unnamed: 0,0
CustomerID,0
ProdTaken,0
Age,0
TypeofContact,0
CityTier,0
DurationOfPitch,0
Occupation,0
Gender,0
NumberOfPersonVisiting,0


Target distribution:


,count
ProdTaken,
0,3331
1,797


Descriptive summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,4128.0,NaN,NaN,NaN,2527.763808,1409.439133,0.0,1320.75,2603.5,3748.25,4887.0
CustomerID,4128.0,NaN,NaN,NaN,202527.763808,1409.439133,200000.0,201320.75,202603.5,203748.25,204887.0
ProdTaken,4128.0,NaN,NaN,NaN,0.193072,0.394757,0.0,0.0,0.0,0.0,1.0
Age,4128.0,NaN,NaN,NaN,37.231831,9.174521,18.0,31.0,36.0,43.0,61.0
TypeofContact,4128,2,Self Enquiry,2918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CityTier,4128.0,NaN,NaN,NaN,1.663275,0.92064,1.0,1.0,1.0,3.0,3.0
DurationOfPitch,4128.0,NaN,NaN,NaN,15.584787,8.398142,5.0,9.0,14.0,20.0,127.0
Occupation,4128,4,Salaried,1999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,4128,3,Male,2463,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NumberOfPersonVisiting,4128.0,NaN,NaN,NaN,2.94937,0.718818,1.0,2.0,3.0,3.0,5.0


## 2. Data Preparation
The raw CSV contains an automatically generated `Unnamed: 0` index and `CustomerID`. These are not useful predictive features, so they are removed.

Known inconsistent category labels are standardized:
- `Fe Male` → `Female`
- `Unmarried` → `Single`

Missing numeric values are imputed with the median and categorical values with the mode. The cleaned data is then split into stratified training and testing sets and saved under `artifacts/`, which acts as the workflow artifact passed to the model-building stage.

In [3]:
from sklearn.model_selection import train_test_split

TARGET = "ProdTaken"
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

clean_df = df.copy()

# Remove unnecessary/non-predictive columns.
clean_df = clean_df.drop(columns=["Unnamed: 0", "CustomerID"], errors="ignore")

# Standardize inconsistent category values.
clean_df["Gender"] = clean_df["Gender"].replace({"Fe Male": "Female"})
clean_df["MaritalStatus"] = clean_df["MaritalStatus"].replace({"Unmarried": "Single"})

# Basic missing-value treatment.
for col in clean_df.select_dtypes(include="number").columns:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in clean_df.select_dtypes(include="object").columns:
    clean_df[col] = clean_df[col].fillna(clean_df[col].mode()[0])

print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)
print("Removed columns:", ["Unnamed: 0", "CustomerID"])
print("Remaining missing values:", int(clean_df.isna().sum().sum()))
display(clean_df.head())


Original shape: (4128, 21)
Cleaned shape: (4128, 19)
Removed columns: ['Unnamed: 0', 'CustomerID']
Remaining missing values: 0


,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,32.0,Company Invited,1,8.0,Salaried,Male,3,3.0,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [4]:
train_df, test_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=42,
    stratify=clean_df[TARGET]
)

train_path = ARTIFACT_DIR / "train.csv"
test_path = ARTIFACT_DIR / "test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Training artifact saved: {train_path} -> {train_df.shape}")
print(f"Testing artifact saved:  {test_path} -> {test_df.shape}")
print("\nTraining target distribution:")
display(train_df[TARGET].value_counts(normalize=True).rename("proportion"))
print("\nTesting target distribution:")
display(test_df[TARGET].value_counts(normalize=True).rename("proportion"))


Training artifact saved: artifacts/train.csv -> (3302, 19)
Testing artifact saved:  artifacts/test.csv -> (826, 19)

Training target distribution:


ProdTaken
0    0.806784
1    0.193216
Name: proportion, dtype: float64


Testing target distribution:


ProdTaken
0    0.807506
1    0.192494
Name: proportion, dtype: float64

## 3. Model Building and Experimentation Tracking
The model-building stage loads **only the train/test workflow artifacts** created above.

A preprocessing pipeline handles:
- Numeric imputation and scaling
- Categorical imputation and one-hot encoding

Four candidate algorithms are tuned using `GridSearchCV`:
- Logistic Regression (baseline)
- Random Forest
- Gradient Boosting
- AdaBoost

The tuned parameters and validation/test metrics are recorded in `artifacts/experiment_results.csv`. The model with the best test F1 score is persisted as `models/best_model.joblib`.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib, json

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=TARGET)
y_train = train_df[TARGET]
X_test = test_df.drop(columns=TARGET)
y_test = test_df[TARGET]

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

experiments = [
    ("LogisticRegression",
     LogisticRegression(max_iter=2000, class_weight="balanced"),
     {"model__C": [0.1, 1.0, 10.0]}),
    ("RandomForest",
     RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1),
     {"model__n_estimators": [150], "model__max_depth": [None, 12],
      "model__min_samples_split": [2, 5]}),
    ("GradientBoosting",
     GradientBoostingClassifier(random_state=42),
     {"model__n_estimators": [100], "model__learning_rate": [0.05, 0.1],
      "model__max_depth": [2, 3]}),
    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     {"model__n_estimators": [100], "model__learning_rate": [0.5, 1.0]})
]

results = []
best_score = -1
best_search = None
best_row = None

for name, estimator, param_grid in experiments:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    search = GridSearchCV(
        pipe, param_grid, cv=3, scoring="f1", n_jobs=-1, refit=True
    )
    search.fit(X_train, y_train)

    predictions = search.predict(X_test)
    probabilities = search.predict_proba(X_test)[:, 1]

    row = {
        "model": name,
        "best_params": json.dumps(search.best_params_),
        "cv_f1": search.best_score_,
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_precision": precision_score(y_test, predictions, zero_division=0),
        "test_recall": recall_score(y_test, predictions, zero_division=0),
        "test_f1": f1_score(y_test, predictions, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, probabilities)
    }
    results.append(row)

    if row["test_f1"] > best_score:
        best_score = row["test_f1"]
        best_search = search
        best_row = row
        best_predictions = predictions
        best_probabilities = probabilities

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)
results_df.to_csv(ARTIFACT_DIR / "experiment_results.csv", index=False)

print("Experimentation tracking — tuned parameters and evaluation metrics:")
display(results_df)


Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 720, in _warm_feature_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 704, in _warm_collaboration_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/generated/interface/models.py", line 30820, in hydrate_crdt_from_proto
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/remote.py", line 749, in __call__
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/client.py", line 150, in call
artifact_tool.rpc.client.RemoteE

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 720, in _warm_feature_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 704, in _warm_collaboration_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/generated/interface/models.py", line 30820, in hydrate_crdt_from_proto
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/remote.py", line 749, in __call__
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/client.py", line 150, in call
artifact_tool.rpc.client.RemoteE

Experimentation tracking — tuned parameters and evaluation metrics:


,model,best_params,cv_f1,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
1,RandomForest,"{""model__max_depth"": null, ""model__min_samples...",0.673879,0.915254,0.893805,0.635220,0.742647,0.964819
2,GradientBoosting,"{""model__learning_rate"": 0.1, ""model__max_dept...",0.536495,0.869249,0.800000,0.427673,0.557377,0.877476
0,LogisticRegression,"{""model__C"": 0.1}",0.532191,0.737288,0.400685,0.735849,0.518847,0.821966
3,AdaBoost,"{""model__learning_rate"": 1.0, ""model__n_estima...",0.436854,0.838983,0.685714,0.301887,0.419214,0.829703


### Best Model Selection
The final model is selected using **test F1 score**, which is appropriate here because the positive class (customers who purchase) is much smaller than the non-purchase class. ROC-AUC is also reported as a secondary metric.

In [6]:
print("BEST MODEL:", best_row["model"])
print("BEST TUNED PARAMETERS:")
print(best_row["best_params"])

print("\nFINAL TEST METRICS")
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    print(f"{metric}: {best_row[metric]:.4f}")

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, best_predictions, digits=4))

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / "best_model.joblib"
joblib.dump(best_search.best_estimator_, model_path)

metrics_to_save = {
    "model": best_row["model"],
    "cv_f1": float(best_row["cv_f1"]),
    "test_accuracy": float(best_row["test_accuracy"]),
    "test_precision": float(best_row["test_precision"]),
    "test_recall": float(best_row["test_recall"]),
    "test_f1": float(best_row["test_f1"]),
    "test_roc_auc": float(best_row["test_roc_auc"]),
    "best_params": json.loads(best_row["best_params"])
}
(ARTIFACT_DIR / "best_metrics.json").write_text(json.dumps(metrics_to_save, indent=2))

print(f"\nBest model saved successfully to: {model_path}")


BEST MODEL: RandomForest
BEST TUNED PARAMETERS:
{"model__max_depth": null, "model__min_samples_split": 5, "model__n_estimators": 150}

FINAL TEST METRICS
test_accuracy: 0.9153
test_precision: 0.8938
test_recall: 0.6352
test_f1: 0.7426
test_roc_auc: 0.9648

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9187    0.9820    0.9493       667
           1     0.8938    0.6352    0.7426       159

    accuracy                         0.9153       826
   macro avg     0.9062    0.8086    0.8460       826
weighted avg     0.9139    0.9153    0.9095       826


Best model saved successfully to: models/best_model.joblib


## 4. Model Deployment — Streamlit
The repository contains `app.py`. The app loads `models/best_model.joblib`, collects customer inputs, converts them into a one-row pandas DataFrame, and returns the purchase prediction and probability.

Deployment files:
- `app.py` — Streamlit user interface
- `requirements.txt` — Python dependencies
- `models/best_model.joblib` — committed trained model

The app is ready to be deployed on **Streamlit Community Cloud** by selecting `app.py` as the application entry point.

In [7]:
# Verify that the saved model can be reloaded and accepts a sample dataframe.
loaded_model = joblib.load(model_path)

sample_customer = X_test.iloc[[0]].copy()
sample_prediction = int(loaded_model.predict(sample_customer)[0])
sample_probability = float(loaded_model.predict_proba(sample_customer)[0, 1])

print("Deployment smoke test")
print("Input shape:", sample_customer.shape)
print("Predicted class:", sample_prediction)
print(f"Purchase probability: {sample_probability:.2%}")
display(sample_customer)


Deployment smoke test
Input shape: (1, 18)
Predicted class: 0
Purchase probability: 6.38%


,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,34.0,Company Invited,1,9.0,Salaried,Male,2,4.0,Basic,3.0,Married,4.0,0,1,0,0.0,Executive,17979.0


## 5. MLOps Pipeline with GitHub Actions
The repository contains `.github/workflows/pipeline.yml`.

The automated workflow performs:
1. Checkout code
2. Set up Python
3. Install dependencies
4. Validate/register dataset
5. Prepare train/test workflow artifacts
6. Train and tune candidate models
7. Upload workflow artifacts
8. Commit the updated model and metrics to the `main` branch

The workflow uses `[skip ci]` on the automated commit to prevent an unnecessary second workflow run.

In [8]:
from pathlib import Path

print("Required repository files:")
required = [
    "data/tourism.csv",
    "src/data_validation.py",
    "src/prepare_data.py",
    "src/train_model.py",
    "models/best_model.joblib",
    "app.py",
    "requirements.txt",
    ".github/workflows/pipeline.yml",
    "README.md"
]
for item in required:
    exists = Path(item).exists()
    print(f"{'✓' if exists else '✗'} {item}")
    assert exists, f"Missing required file: {item}"

print("\nProject validation: PASSED")


Required repository files:
✓ data/tourism.csv
✓ src/data_validation.py
✓ src/prepare_data.py
✓ src/train_model.py
✓ models/best_model.joblib
✓ app.py
✓ requirements.txt
✓ .github/workflows/pipeline.yml
✓ README.md

Project validation: PASSED


## 6. Submission Checklist

| Requirement | Evidence in project |
|---|---|
| Data Registration | `data/tourism.csv` + schema validation |
| Data Preparation | Cleaning + `artifacts/train.csv` + `artifacts/test.csv` |
| Workflow Artifact | `artifacts/` directory and GitHub Actions upload-artifact |
| Model Building | 4 algorithms + GridSearchCV |
| Experimentation Tracking | `artifacts/experiment_results.csv` with tuned parameters |
| Evaluation | Accuracy, Precision, Recall, F1, ROC-AUC + classification report |
| Best Model | `models/best_model.joblib` |
| Streamlit Deployment | `app.py` + `requirements.txt` |
| GitHub Actions | `.github/workflows/pipeline.yml` |
| Automated Model Update | Workflow commits model/metrics to `main` |
| Documentation | `README.md` |
